# 15 · Prompting the harness: program.md & results.tsv as memory

The agent's context each iteration is three artifacts: **program.md** (immutable
direction, system prompt, prompt-cached), **results.tsv** (episodic memory), and
the **modifiable source**. We assemble the actual prompt — no API call needed.

In [1]:
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parent))
import matplotlib; matplotlib.use("Agg")
import numpy as np, matplotlib.pyplot as plt
from harness.data import make_synthetic_dataset, DatasetSpec, load_split, class_names

DATA = Path("../data/bdd-tiny.lance")
if not DATA.exists():
    make_synthetic_dataset(DATA, DatasetSpec(n=3000, seed=7))
print("dataset:", DATA, "| NOTE: these chapters scale to bdd-small/full; here we")
print("demonstrate the mechanics on the tiny tier so they run with no GPU/cluster.")

dataset: ../data/bdd-tiny.lance | NOTE: these chapters scale to bdd-small/full; here we
demonstrate the mechanics on the tiny tier so they run with no GPU/cluster.


In [2]:
from harness.loop import run_loop
run_loop(iters=4, dataset_path=str(DATA), results_path="results.tsv", verbose=False)

system_prompt = Path("../program.md").read_text()
print("SYSTEM PROMPT: program.md  (", len(system_prompt), "chars, prompt-cached )\n")
print(system_prompt[:380], "...\n")

SYSTEM PROMPT: program.md  ( 3051 chars, prompt-cached )

# program.md — agent direction

> This file is **human-authored** and **immutable during a run**. It is the
> only place where the goal of the loop is stated. The harness reads it once at
> startup. The agent may *read* it as often as it likes but must never edit it.

## Objective

Maximize the **official score** produced by `harness/evaluator.py` on the
held-out `test` split o ...



In [3]:
import csv
rows = list(csv.DictReader(open("results.tsv"), delimiter="\t"))
def format_history(rows, last=6):
    best = max(rows, key=lambda r: float(r["score"]))
    lines = [f"best so far: score={best['score']} via {best['action']}"]
    lines.append("recent attempts (compress older ones at scale):")
    for r in rows[-last:]:
        lines.append(f"  iter {r['iter']}: score={r['score']} kept={r['kept']} "
                     f"worst={r['worst_group_name']}({r['worst_group']}) :: {r['rationale']}")
    return "\n".join(lines)
print("USER MESSAGE (episodic memory from results.tsv):\n")
print(format_history(rows))

USER MESSAGE (episodic memory from results.tsv):

best so far: score=1.0179 via {'model': {'width': 32}}
recent attempts (compress older ones at scale):
  iter 0: score=0.8835 kept=True worst=foggy(0.6444) :: baseline config
  iter 1: score=0.9718 kept=True worst=foggy(0.7333) :: Fog is the worst, rarest slice -> upweight fog frames 5x.
  iter 2: score=0.9745 kept=True worst=foggy(0.7333) :: Expand fog upweighting to LanceDB nearest neighbours (fog-like frames).
  iter 3: score=1.0179 kept=True worst=foggy(0.7778) :: Give the net more capacity to fit the weak fog signal.
  iter 4: score=0.9512 kept=False worst=foggy(0.7111) :: Spend more of the budget on epochs to extract the weak fog cue.


In [4]:
PROPOSE_CHANGE_TOOL = {
    "name": "propose_change",
    "description": "Propose one config override to try next, with a rationale.",
    "input_schema": {"type": "object", "properties": {
        "override": {"type": "object"},
        "rationale": {"type": "string"}}, "required": ["override", "rationale"]}}
print("TOOL the model calls to return a structured proposal:")
import json; print(json.dumps(PROPOSE_CHANGE_TOOL, indent=2)[:400], "...")

TOOL the model calls to return a structured proposal:
{
  "name": "propose_change",
  "description": "Propose one config override to try next, with a rationale.",
  "input_schema": {
    "type": "object",
    "properties": {
      "override": {
        "type": "object"
      },
      "rationale": {
        "type": "string"
      }
    },
    "required": [
      "override",
      "rationale"
    ]
  }
} ...


A good `rationale` names the hypothesis and the slice — it's the breadcrumb the
next iteration reads. As history grows, compress the middle (keep the Pareto
frontier + the last few attempts in full) to stay inside the context budget.